# Февраль 2026: атрибуты дашборда (минимум комментариев)


In [ ]:
import re
import warnings
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

warnings.filterwarnings('ignore', category=FutureWarning)

pd.options.display.max_columns = None
pd.options.display.width = None


In [ ]:
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
aur_rate = 1926.0

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_merchants')
    imp.execute('invalidate metadata ods_alpha.scd1_pos_terminals')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_int')
    imp.execute('invalidate metadata ods.scd1_z_r2_ip_merchants')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_tune')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_fix')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_plan')
    imp.execute('invalidate metadata ods.scd1_z_depart')
    imp.execute('invalidate metadata ods.scd1_z_branch')
    imp.execute('invalidate metadata ods.scd1_z_cl_corp')
    imp.execute('invalidate metadata sandbox_ai.shestopalov_terminal_amortization_oneoff')

def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    return s if s else None

def normalize_inn(v):
    s = normalize_numeric_str(v)
    if s is None:
        return None
    if len(s) in (10, 12):
        return s
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return None

def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s.lower()

def normalize_agr_id(v):
    return normalize_numeric_str(v)


In [ ]:
def to_sql_in_list(values):
    vals = [str(v) for v in values if pd.notna(v) and str(v).strip() != '']
    if not vals:
        return None
    return ', '.join([f"'{v}'" for v in vals])

# base
sql_base = f"""
select
    case
      when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 9 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 10, '0')
      when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 11 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 12, '0')
      else regexp_replace(trim(c.c_inn), '[^0-9]', '')
    end as inn_norm,
    cast(c.c_cmp_name as string) as client_name,
    lower(trim(cast(a.c_agr_number as string))) as contract_norm,
    cast(a.c_agr_number as string) as contract_number,
    cast(a.abs_agr_id as string) as agr_id_norm,
    cast(a.n_agr as string) as n_agr,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to,
    a.n_cmp_client
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
where a.acq_class = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
"""
with imp:
    base_df = imp.fetch(sql_base)

base_df['inn_norm'] = base_df['inn_norm'].apply(normalize_inn)
base_df['contract_norm'] = base_df['contract_norm'].apply(normalize_contract)
base_df['agr_id_norm'] = base_df['agr_id_norm'].apply(normalize_agr_id)

agr_ids = sorted(base_df['agr_id_norm'].dropna().unique().tolist())
n_agrs = sorted(base_df['n_agr'].dropna().unique().tolist())
n_cmps = sorted(base_df['n_cmp_client'].dropna().astype(str).unique().tolist())

# r2 attrs
r2_chunks = []
for i in range(0, len(agr_ids), 500):
    part = agr_ids[i:i+500]
    in_list = to_sql_in_list(part)
    sql = f"""
    with r2 as (
        select cast(m.id as string) as agr_id_norm, m.c_cl_org, m.c_depart, m.c_tariff_plan
        from ods.scd1_z_r2_ip_merchants m
        where cast(m.id as string) in ({in_list})
    ),
    fix_r2 as (
        select cast(m.id as string) as agr_id_norm,
               max(cast(tf.c_summa as double)) as commission_monthly,
               count(distinct cast(tf.c_summa as string)) as fix_variants_cnt
        from ods.scd1_z_r2_ip_merchants m
        left join ods.scd1_z_r2_tariff_tune tt on tt.c_tariff_plan = m.c_tariff_plan
        left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
        where cast(m.id as string) in ({in_list})
        group by cast(m.id as string)
    )
    select r2.agr_id_norm,
           corp.c_register_gos_reg_num_rec as ogrn,
           dep.c_name as vsp_name,
           dep.c_num as vsp_code,
           br.c_shortlabel as filial_rf,
           tp.c_name as tariff_name,
           fr.commission_monthly,
           fr.fix_variants_cnt
    from r2
    left join ods.scd1_z_cl_corp corp on corp.id = r2.c_cl_org
    left join ods.scd1_z_depart dep on dep.id = r2.c_depart
    left join ods.scd1_z_branch br on br.id = dep.c_filial
    left join ods.scd1_z_r2_tariff_plan tp on tp.id = r2.c_tariff_plan
    left join fix_r2 fr on fr.agr_id_norm = r2.agr_id_norm
    """
    with imp:
        df = imp.fetch(sql)
        if len(df) > 0:
            r2_chunks.append(df)
r2_df = pd.concat(r2_chunks, ignore_index=True) if r2_chunks else pd.DataFrame()

# company stats
cmp_chunks = []
for i in range(0, len(n_cmps), 300):
    part = n_cmps[i:i+300]
    in_list = ', '.join(part)
    sql = f"""
    with cmp_retl as (
        select m.n_cmp as n_cmp_client,
               count(distinct cast(m.c_nmrc as string)) as retl_cnt,
               min(cast(m.n_mcc as string)) as mcc_any,
               count(distinct cast(m.n_mcc as string)) as mcc_variants_cnt
        from ods_alpha.scd1_merchants m
        where m.n_cmp in ({in_list}) and m.c_nmrc is not null
        group by m.n_cmp
    ),
    cmp_terms as (
        select distinct m.n_cmp as n_cmp_client,
               cast(t.c_nter as string) as c_nter
        from ods_alpha.scd1_merchants m
        join ods_alpha.scd1_pos_terminals t on t.c_nmrc = m.c_nmrc and t.c_nter is not null
        where m.n_cmp in ({in_list}) and m.c_nmrc is not null
    ),
    cmp_term_agg as (
        select n_cmp_client, count(distinct c_nter) as term_cnt
        from cmp_terms
        group by n_cmp_client
    ),
    cmp_amort as (
        select ct.n_cmp_client,
               sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
        from cmp_terms ct
        left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
          on cast(am.c_nter as string) = ct.c_nter
        group by ct.n_cmp_client
    )
    select coalesce(r.n_cmp_client, t.n_cmp_client, a.n_cmp_client) as n_cmp_client,
           r.retl_cnt,
           r.mcc_any,
           r.mcc_variants_cnt,
           t.term_cnt,
           a.amortization
    from cmp_retl r
    full outer join cmp_term_agg t on t.n_cmp_client = r.n_cmp_client
    full outer join cmp_amort a on a.n_cmp_client = coalesce(r.n_cmp_client, t.n_cmp_client)
    """
    with imp:
        df = imp.fetch(sql)
        if len(df) > 0:
            cmp_chunks.append(df)
cmp_df = pd.concat(cmp_chunks, ignore_index=True) if cmp_chunks else pd.DataFrame()

# trx stats (very memory-safe mode)
trx_chunks = []
for i in range(0, len(n_agrs), 80):
    part = n_agrs[i:i+80]
    in_list = to_sql_in_list(part)

    sql_acq = f"""
    select ta.n_trx,
           cast(ta.n_agr as string) as n_agr,
           coalesce(cast(ta.n_amt_tax as double), 0.0) as n_amt_tax
    from ods_alpha.scd1_trx_acq ta
    where cast(ta.n_agr as string) in ({in_list})
    """
    with imp:
        ta_df = imp.fetch(sql_acq)

    if len(ta_df) == 0:
        continue

    ta_df['n_trx'] = ta_df['n_trx'].astype(str)
    trx_ids = sorted(ta_df['n_trx'].dropna().unique().tolist())

    trx_base_chunks = []
    for j in range(0, len(trx_ids), 1200):
        tr_part = trx_ids[j:j+1200]
        tr_in = to_sql_in_list(tr_part)
        sql_trx = f"""
        select cast(t.n_trx as string) as n_trx,
               cast(t.n_amt_src as double) as n_amt_src
        from ods_alpha.scd1_trx t
        where cast(t.n_trx as string) in ({tr_in})
          and to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
          and t.c_nter is not null
          and coalesce(t.ods_deleted_flg, '0') <> '1'
        """
        with imp:
            d = imp.fetch(sql_trx)
            if len(d) > 0:
                trx_base_chunks.append(d)

    if not trx_base_chunks:
        continue
    tb_df = pd.concat(trx_base_chunks, ignore_index=True)

    int_chunks = []
    for j in range(0, len(trx_ids), 1200):
        tr_part = trx_ids[j:j+1200]
        tr_in = to_sql_in_list(tr_part)
        sql_int = f"""
        select cast(i.n_trx as string) as n_trx,
               coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee
        from ods_alpha.scd1_trx_int i
        where cast(i.n_trx as string) in ({tr_in})
        """
        with imp:
            d = imp.fetch(sql_int)
            if len(d) > 0:
                int_chunks.append(d)

    int_df = pd.concat(int_chunks, ignore_index=True) if int_chunks else pd.DataFrame(columns=['n_trx', 'n_amt_fee'])

    tmp = ta_df.merge(tb_df, on='n_trx', how='inner')
    tmp = tmp.merge(int_df, on='n_trx', how='left')
    tmp['n_amt_fee'] = pd.to_numeric(tmp['n_amt_fee'], errors='coerce').fillna(0)

    g = tmp.groupby('n_agr', as_index=False).agg(
        trx_cnt=('n_trx', 'nunique'),
        trx_sum=('n_amt_src', 'sum'),
        commission_from_ops=('n_amt_tax', 'sum'),
        int_component=('n_amt_fee', 'sum')
    )
    trx_chunks.append(g)

trx_df = pd.concat(trx_chunks, ignore_index=True) if trx_chunks else pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component'])
if len(trx_df) > 0:
    trx_df = trx_df.groupby('n_agr', as_index=False).agg(
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
        int_component=('int_component', 'sum')
    )

# merge + formulas
attrs_df = base_df.copy()
if len(r2_df):
    attrs_df = attrs_df.merge(r2_df, on='agr_id_norm', how='left')
if len(cmp_df):
    attrs_df['n_cmp_client'] = attrs_df['n_cmp_client'].astype(str)
    cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)
    attrs_df = attrs_df.merge(cmp_df, on='n_cmp_client', how='left')
if len(trx_df):
    attrs_df = attrs_df.merge(trx_df, on='n_agr', how='left')

for c in ['retl_cnt','term_cnt','trx_cnt','trx_sum','amortization','commission_from_ops','commission_monthly','int_component']:
    if c in attrs_df.columns:
        attrs_df[c] = pd.to_numeric(attrs_df[c], errors='coerce').fillna(0)

attrs_df['aur'] = pd.to_numeric(attrs_df.get('term_cnt', 0), errors='coerce').fillna(0) * aur_rate
attrs_df['commission_total'] = attrs_df.get('commission_from_ops', 0) + attrs_df.get('commission_monthly', 0)
attrs_df['nod_te'] = attrs_df['commission_total'] - attrs_df.get('int_component', 0)
attrs_df['fin_result_te'] = attrs_df['nod_te'] - attrs_df['aur'] - attrs_df.get('amortization', 0)
attrs_df['avg_acq_pct'] = attrs_df.apply(lambda r: (r['commission_from_ops'] / r['trx_sum'] * 100.0) if pd.notna(r['trx_sum']) and r['trx_sum'] not in (0, 0.0) else None, axis=1)
attrs_df['avg_irf_pct'] = attrs_df.apply(lambda r: (r['int_component'] / r['trx_sum'] * 100.0) if pd.notna(r['trx_sum']) and r['trx_sum'] not in (0, 0.0) else None, axis=1)

attrs_df['inn_norm'] = attrs_df['inn_norm'].apply(normalize_inn)
attrs_df['contract_norm'] = attrs_df['contract_norm'].apply(normalize_contract)
attrs_df['agr_id_norm'] = attrs_df['agr_id_norm'].apply(normalize_agr_id)

display(pd.DataFrame([
    {'metric': 'rows_total', 'value': len(attrs_df)},
    {'metric': 'unique_inn_contract', 'value': attrs_df[['inn_norm', 'contract_norm']].drop_duplicates().shape[0]},
    {'metric': 'rows_without_agr_id', 'value': int(attrs_df['agr_id_norm'].isna().sum()) if 'agr_id_norm' in attrs_df.columns else None},
]))
display(attrs_df.head(30))